# Nori's Refined Mood Engine
Statistical analysis of Argo float temperature data against the CalCOFI historical baseline.

---

## Background & Context

### Who is Nori?
**Nori** is an **Argo float** — a small, autonomous underwater robot (ID: 3901161) deployed in the Pacific Ocean. She has no propulsion; she drifts with ocean currents and controls her depth by changing her buoyancy. Every ~10 days she:
1. Sinks to ~2000m depth
2. Drifts at depth collecting data
3. Rises back to the surface, transmitting temperature, salinity, and pressure readings via satellite

We give her a name and a "mood" to make her data human-readable and to drive an automated decision: **should she dive or stay near the surface?**

---

### What is the CalCOFI Baseline?
**CalCOFI** (California Cooperative Oceanic Fisheries Investigations) is a long-running ocean monitoring program off the California coast. Their historical temperature records give us a **reference mean of 11.536°C** — what "normal" Pacific Ocean temperature looks like averaged across depth and time.

This baseline is our **anchor**. Any temperature Nori measures is compared against it to decide if the ocean is unusually warm, unusually cold, or normal.

---

### What is a Cycle?
Each time Nori completes one full dive-and-resurface trip, that is one **cycle** (tracked by `meta_cycle_number`). Each cycle produces dozens of temperature readings at different depths (pressures). This notebook analyzes each cycle as a group to ask: *was this cycle's water statistically different from the historical norm?*

---

### Key Relationships in the Data

| Column | What it means |
|---|---|
| `pressure (decibar)` | Depth proxy — higher pressure = deeper water. ~10 decibar ≈ 10m depth |
| `temperature (degree_celsius)` | In-situ water temperature Nori measured at that depth |
| `Temperature Anomaly` | `temperature - 11.536` — how far from the CalCOFI normal |
| `meta_cycle_number` | Which dive trip this reading came from |
| `Mood` | Derived label: is Nori in water that's too hot, too cold, or just right? |
| `p_value` | From a one-sample t-test — is the anomaly real or just random noise? |

---

### The Core Question
> *If Nori is Sweating 🥵 at the surface (water is significantly warmer than the historical baseline), does diving deeper bring her back to a Happy 😊 state?*

The graph below answers this: as pressure increases (deeper water), temperatures drop toward the baseline, and Nori's mood shifts from Sweating → Happy. This makes **"Dive"** the statistically defensible command when surface anomaly is confirmed at p < 0.05.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

HISTORICAL_MEAN = 11.536

## 1. Data Loading

In [ ]:
COLS = [
    'pressure (decibar)',
    'temperature (degree_celsius)',
    'meta_cycle_number'
]

df = pd.read_csv('single_argo.csv', usecols=COLS)
df = df.dropna(subset=['temperature (degree_celsius)'])

print(f"Loaded {len(df)} rows across {df['meta_cycle_number'].nunique()} cycles")
df.head()

## 2. Temperature Anomaly & Mood

In [ ]:
df['Temperature Anomaly'] = df['temperature (degree_celsius)'] - HISTORICAL_MEAN

def assign_mood(anomaly):
    if anomaly > 0:
        return 'Sweating \U0001f975'
    elif anomaly < 0:
        return 'Shivering \U0001f976'
    else:
        return 'Happy \U0001f60a'

df['Mood'] = df['Temperature Anomaly'].apply(assign_mood)
df[['pressure (decibar)', 'temperature (degree_celsius)', 'Temperature Anomaly', 'Mood']].head(10)

## 3. The Statistical Mood Engine

In [ ]:
def get_ocean_status(group):
    temps = group['temperature (degree_celsius)'].dropna()
    mean_temp = temps.mean()
    anomaly = mean_temp - HISTORICAL_MEAN

    t_stat, p_value = stats.ttest_1samp(temps, HISTORICAL_MEAN)

    if anomaly > 0 and p_value < 0.05:
        mood = 'Sweating \U0001f975'
        confidence = 'Statistically Significant'
    elif anomaly < 0 and p_value < 0.05:
        mood = 'Shivering \U0001f976'
        confidence = 'Statistically Significant'
    else:
        mood = 'Happy \U0001f60a'
        confidence = 'Normal/Inconclusive'

    return pd.Series({
        'mean_temperature': round(mean_temp, 3),
        'anomaly': round(anomaly, 3),
        'p_value': round(p_value, 4),
        'Mood': mood,
        'Confidence': confidence
    })

## 4. Data Analytics Track — Cycle Summary

In [ ]:
summary = df.groupby('meta_cycle_number').apply(get_ocean_status).reset_index()
print(f"Summary across {len(summary)} cycles:")
summary

## Researcher Decision Framework

Each row above represents one dive cycle. The **Mood** and **Confidence** columns power the following decision rule:

> **If the Mood is 'Sweating' AND Confidence is 'Statistically Significant'**, the data-driven decision is to **command an immediate dive** to preserve sensor health. A positive anomaly (mean cycle temperature above the CalCOFI baseline of 11.536 °C) that passes the one-sample t-test at α = 0.05 means the warming is real — not random noise — and warrants action. Descending to cooler, more stable deep-ocean layers restores the float to its 'Happy' operating range and ensures continued scientific validity of the profile data.

A p-value < 0.05 is the sole statistical gate: the direction of the anomaly (positive = Sweating, negative = Shivering) determines which action to take.

## 5. Visualization — Temperature vs. Pressure (The Climate Gap)

In [ ]:
mood_colors = {
    'Sweating \U0001f975': '#e74c3c',
    'Happy \U0001f60a': '#2ecc71',
    'Shivering \U0001f976': '#3498db'
}

fig, ax = plt.subplots(figsize=(10, 7))

for mood, color in mood_colors.items():
    mask = df['Mood'] == mood
    ax.scatter(
        df.loc[mask, 'temperature (degree_celsius)'],
        df.loc[mask, 'pressure (decibar)'],
        c=color, alpha=0.5, s=15, label=mood
    )

ax.axvline(x=HISTORICAL_MEAN, color='red', linewidth=2.5,
           linestyle='--', label=f'CalCOFI Baseline ({HISTORICAL_MEAN}°C)')

ax.invert_yaxis()
ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('Pressure (decibar) — deeper →', fontsize=12)
ax.set_title("Nori's Mood vs. Depth\nProving that Diving leads to a Happy 😊 state", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('nori_mood_vs_depth.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as nori_mood_vs_depth.png")

## 6. Mood Distribution Summary

In [ ]:
print("=== Row-level Mood Distribution ===")
print(df['Mood'].value_counts())
print()
print("=== Cycle-level Mood Distribution ===")
print(summary['Mood'].value_counts())